# Getting Started v2

This is the second version (actually 14th), versions 1-6 can be classified as the first version of the notebook since there was not any major update. Now this can be a bit more complex. Version 7 onwards, shall be collectively considered second version. And anybody having difficulty understanding something, start with earlier version.

**What's in here??**

* A basic structure of how things are to be done
* Nothing complex is done, it's your task to explore further complex topics on your own

**What's new?**

* Using XGBoost Over predictions of Logistic Regression (stacking)

This can be used as a reference for those who are just getting started and get stuck on things like, why is there a error during submission?

**The common issues with beginners**

1. My CV score is high but it is comparatively very low on public lb.
     1. People often struggle with roc_auc score as metric, thinking it's classification, discrete values are submitted.
     2. But, the competition asks for the probabilities, hence while making predictions - instead of model.predict, we use model.predict_proba
2. Many new commers put the wrong column labels in the submission file.
3. There may me other issues too... So yeah, we have the discussion forum for that.

**To the talented people out there! Please share your comments and tell if I am missing something.**

# Importing Libraries Used

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression # Classification Model
from sklearn.model_selection import cross_val_score, KFold # Model Evaluation
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore") # ignore, lol

print("="*20)
print("Required Libraries Imported")
print("="*20)

Required Libraries Imported


# Loading Data

In [2]:
# Loading Data
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv') 
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv')

# Setting the target - category to 0/1
train['Heart Disease']=train['Heart Disease'].map({'Absence':0, 'Presence':1})
train.drop('id', axis=1, inplace=True) # id is not used for model training and prediction
testID=test.pop('id') # important to preserve test IDs for final submission.
y=train.pop('Heart Disease') # TARGET

print("="*20)
print("Loaded Successfully!!")
print("="*20)

Loaded Successfully!!


# One-Hot-Encoding

In [3]:
# One Hot All columns 
# (Selective Encoding can be done, but that does not help for the current data)
# Reason :- https://www.kaggle.com/code/rattans/logistic-regression-ps-s6e2/comments#3401628

oh_cols = train.columns.to_list()
oh = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

oh_train = pd.DataFrame(oh.fit_transform(train[oh_cols]))
train[oh.get_feature_names_out(train[oh_cols].columns)] = oh_train.astype(int) #.astype('category')

oh_test= pd.DataFrame(oh.transform(test[oh_cols]))
test[oh.get_feature_names_out(test[oh_cols].columns)] = oh_test.astype(int) #.astype('category')

print("="*20)
print("Encoding Completed")
print("="*20)

Encoding Completed


# Scaling the Data

In [4]:
# Speed of Logistic Regression Increases if the data is scaled properly
# (results shall also increase, but speed is more observable)
scaler = StandardScaler() # Just know that this scales the data
scaled_train = pd.DataFrame(scaler.fit_transform(train))
scaled_train.columns = train.columns
scaled_test = pd.DataFrame(scaler.transform(test))
scaled_test.columns = test.columns

print("="*20)
print("Data Scaled")
print("="*20)

Data Scaled


# Logistic Regression

In [5]:
print("="*20)
print("Training Logistic Regression...")
model = LogisticRegression() # Our Model, simple one - Logistic Regression

kf=KFold(n_splits=5)
train_label=pd.Series(index=scaled_train.index)

for train_index, test_index in kf.split(scaled_train):
    model.fit(scaled_train.iloc[train_index], y.iloc[train_index])
    pred=model.predict_proba(scaled_train.iloc[test_index])[:,1]
    train_label.iloc[test_index]=pred
    
model.fit(scaled_train, y)
test_label=pd.Series(model.predict_proba(scaled_test)[:,1])
scaled_train['label']=train_label
scaled_test['label']=test_label

print("="*20)
print("Added Logistic Regression predictions to data")
print("="*20)

Training Logistic Regression...
Added Logistic Regression predictions to data


# XGBoost

In [6]:
print("="*20)
print("Final Model = XGBClassifier")
print("="*20)

model2=XGBClassifier(n_estimators=1200,subsample = 0.7973179062433801,
                     colsample_bytree = 0.7369809154998354, 
                     max_depth=3, device='cuda', learning_rate=0.01, random_state=6)
score = cross_val_score(model2, scaled_train, y, cv=5, scoring='roc_auc') # CV Score
print("CV:",score.mean())

Final Model = XGBClassifier
CV: 0.9555942536713034


# Final Training and Making predictions

In [7]:
print("="*20)
print("Final Training")

model2.fit(scaled_train, y) # Final/Full training
pred = model2.predict_proba(scaled_test)[:,1] # predicts the probabilities

print("="*20)
print("Final Predictions made")
print("="*20)

Final Training
Final Predictions made


# Submission

In [8]:
final = pd.DataFrame({'id':testID, 'Heart Disease':pred})
final.to_csv('submission.csv', index=False) # submission file

print("="*20)
print("Submitted Successfully!")
print("="*20)

Submitted Successfully!


# Improvement from Scaling

**Earlier versions of the notebook where there was only Logistic Regression, took 10 minutes to run, but this just takes 2 minutes which include scaling and model training both along with final xgboost training.**

# Moving Forward

So if you are a true begineer, still there are two possibilities!

1. You are new to ML - Then you should take some time learning, Kaggle Learn is also a great place to learn hands-on
2. You are new to Kaggle (not ML) - Then probably this notebook helps you with a slight tweak to how things go on kaggle. Have a great journey exploring things further.

If this helped - consider giving an upvote

# Check this out

Moving further, check this [notebook](https://www.kaggle.com/code/rattans/logistic-regression-ps-s6e2).

I have used one-hot encoding to encode all the features and Logistic Regression performed even better with CV:0.95550 and Public LB: 0.95371

Link : [https://www.kaggle.com/code/rattans/logistic-regression-ps-s6e2](https://www.kaggle.com/code/rattans/logistic-regression-ps-s6e2)

Do check out once.